# Murmurations, Mestre-Nagao sums, and CNNs for Elliptic Curves

This notebook reproduces all figures from the paper
[arXiv:2603.17681](https://arxiv.org/abs/2603.17681).

We train a 1D CNN to predict the analytic rank of elliptic curves from their
Fourier coefficients $a_p$, then use saliency analysis to interpret what the
network learns — revealing connections between murmurations and Mestre-Nagao sums.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from scripts.data import get_prime_columns, CONDUCTOR_RANGES
from scripts.experiments import train_all_ranges, sweep_n_primes, train_saliency_evolution
from scripts.synthetic import generate_fake_ap, classify_fake_ap

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

PRIME_COLUMNS = get_prime_columns(10000)
print(f'Number of prime features: {len(PRIME_COLUMNS)}')

## 1. The XECQ Dataset (Figure 1)

The dataset **XECQ** contains ~1.5M elliptic curves over $\mathbb{Q}$ with conductor $\leq 400{,}000$.
Each curve has analytic rank in $\{0, 1, 2, 3, 4\}$ and is represented by the vector
$v_E = (\tilde{a}_p(E))_{p \leq 10000}$ of normalised Fourier coefficients at the 1229 primes up to 10000.

We work with four conductor sub-intervals of roughly equal size (~37k curves each).

In [ ]:
# Figure 1: Distribution of conductors in XECQ
import pandas as pd, gc
df_cond = pd.read_csv('ECQ6_DF_ap.csv', usecols=['conductor'])
conductors = df_cond['conductor'].to_numpy()
del df_cond; gc.collect()

plt.figure(figsize=(10, 6))
sns.histplot(conductors, bins=100)
plt.title('Distribution of Conductor')
plt.xlabel('Conductor')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

## 2. CNN Architecture and Training (Figure 2, Table 2)

The 1D CNN has three convolutional layers (16 → 32 → 64 channels, kernel size 3),
each followed by ReLU and max-pooling, then dropout (0.5) and two fully connected
layers (128 units). Trained with cross-entropy loss, Adam optimizer (lr=0.001),
batch size 3000.

For each of the four conductor intervals, we split 80:20 into train/test and
train for 100 epochs. We also compute the averaged saliency scores $w_p$ after
training, which will be used for Figures 4-5.

In [ ]:
seed = 11167297796775735125
torch.manual_seed(seed)
np.random.seed(1234)

results = train_all_ranges('ECQ6_DF_ap.csv', PRIME_COLUMNS, device)

In [ ]:
# Figure 2: Test accuracy vs epoch for each conductor range
plt.figure(figsize=(10, 6))
for label in CONDUCTOR_RANGES:
    epochs = len(results[label]['test_acc'])
    plt.plot(range(1, epochs + 1), results[label]['test_acc'], label=label)
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Test Accuracy over Epochs for Different Conductor Range')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Table 2: Best observed accuracy per conductor interval
print('\nBest observed accuracy (Table 2):')
for label in CONDUCTOR_RANGES:
    best = max(results[label]['test_acc'])
    print(f'  {label}: {best:.2%}')

## 3. Accuracy vs Number of Primes (Figure 3)

How many primes does the CNN need to achieve high accuracy? For each conductor interval,
we sweep $\pi(b)$ from 5 to 1229 (step 10) and train with early stopping (patience 10,
max 400 epochs). The result shows that curves with larger conductor require more primes.

This cell loads pre-computed results from `test_accuracies/` if available, or recomputes
(which is slow).

In [ ]:
accuracy_vs_primes = sweep_n_primes('ECQ6_DF_ap.csv', PRIME_COLUMNS, device)

In [ ]:
# Figure 3: Accuracy vs number of primes (linear and log scale)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for label in CONDUCTOR_RANGES:
    primes, accs = accuracy_vs_primes[label]
    ax1.plot(primes, accs, label=label)
    ax2.plot(primes, accs, label=label)

ax1.set_xlabel('Number of primes used')
ax1.set_ylabel('Accuracy')
ax1.set_title('Test Accuracy vs Number of Primes')
ax1.legend()
ax1.grid(True)

ax2.set_xlabel('Number of primes used (log scale)')
ax2.set_ylabel('Accuracy')
ax2.set_xscale('log')
ax2.set_title('Test Accuracy vs Number of Primes (log scale)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## 4. Saliency Curves (Figures 4-5)

To interpret the CNN predictions, we compute **saliency scores** $w_p$ — the
gradient of the predicted class score with respect to input $\tilde{a}_p$,
averaged over test samples.

Comparing $w_p$ with the Mestre-Nagao weighting $\log(p)/\sqrt{p}$ reveals that
the CNN independently discovers weights similar to those in the classical
Mestre-Nagao sum.

In [ ]:
prime_numbers = np.array([float(p) for p in PRIME_COLUMNS])
mestrenagao = np.log(prime_numbers) / np.sqrt(prime_numbers)

# Figure 4: Saliency scores w_p vs Mestre-Nagao weighting
plt.figure(figsize=(10, 6))
for label in CONDUCTOR_RANGES:
    sal = results[label]['saliency']
    plt.scatter(prime_numbers, sal, label=label, s=1)
plt.scatter(prime_numbers, mestrenagao / (np.log(2) / 2) / 5,
            label=r'$\log(p)/\sqrt{p}$ (scaled)', s=1, c='black')
plt.xlabel('p')
plt.ylabel(r'Saliency $w_p$')
plt.title('Saliency scores vs Mestre-Nagao weighting')
plt.legend(markerscale=5)
plt.grid(True)
plt.tight_layout()
plt.show()

# Figure 5: Normalized saliency
plt.figure(figsize=(10, 6))
for label in CONDUCTOR_RANGES:
    sal = results[label]['saliency']
    plt.scatter(prime_numbers, sal / np.max(sal), label=label, s=1)
plt.scatter(prime_numbers, mestrenagao / (np.log(2) / 2) / 8,
            label=r'$\log(p)/\sqrt{p}$ (scaled)', s=1, c='black')
plt.xlabel('p')
plt.ylabel(r'Normalized saliency $\widetilde{w}_p$')
plt.title('Normalized saliency scores vs Mestre-Nagao weighting')
plt.legend(markerscale=5)
plt.grid(True)
plt.tight_layout()
plt.show()

## 5. Per-Class Saliency Evolution (Figures 6-9)

To understand *how* the CNN distinguishes ranks, we compute per-class saliency
$W_p^v$ — the signed, class-specific gradient averaged over samples
predicted as rank $v$ and normalized to $[-1, 1]$.

We save model checkpoints at every training step and plot the saliency curves at
step 0 of each epoch. For the low-conductor range $[0, 10000]$, the CNN first
learns murmurations (opposite signs for ranks 0 and 1) and later shifts to
Mestre-Nagao-like weighting on small primes. For larger conductors, the
murmuration signal diminishes and small-prime weighting dominates from the start.

Each conductor interval produces one figure (Figs 6-9).

In [ ]:
# Figures 6-9: Train and plot saliency evolution for each conductor range
prime_numbers = [float(p) for p in PRIME_COLUMNS]

for label, (row_start, row_end) in CONDUCTOR_RANGES.items():
    print(f'\n=== Saliency evolution for {label} ===')
    grids = train_saliency_evolution(
        'ECQ6_DF_ap.csv', label, row_start, row_end, PRIME_COLUMNS, device,
    )

    n_plots = len(grids)
    n_cols = 4
    n_rows = (n_plots + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    axes = axes.flatten()

    for idx, grid in enumerate(grids):
        ax = axes[idx]
        for v in sorted(grid['class_saliency'].keys()):
            ax.scatter(prime_numbers, grid['class_saliency'][v],
                       s=0.3, c=f'C{v}', label=str(v))
        ax.set_ylim(-1, 1)
        acc_str = f', acc={grid["accuracy"]:.3f}' if grid['accuracy'] else ''
        ax.set_title(f'Epoch {grid["epoch"]}{acc_str}', fontsize=8)
        ax.tick_params(labelsize=6)
        if idx == 0:
            ax.legend(fontsize=6, markerscale=5, title='Rank')

    for idx in range(n_plots, len(axes)):
        axes[idx].set_visible(False)

    fig.suptitle(f'Saliency evolution for {label}', fontsize=14)
    plt.tight_layout()
    plt.show()

## 6. Synthetic Data Experiment (Figure 10)

Code adapted from `generate_fake_ap.py` (Sato-Tate sampling via rejection sampling)
and `classify_and_plot_fake_ranks.py` (classification and murmuration plotting).

We generate 500,000 synthetic sequences by drawing each $a_p$ independently from
the **Sato-Tate distribution** (PDF: $\frac{2}{\pi}\sin^2\theta$ for the angle,
giving $\tilde{a}_p = \cos\theta \in [-1, 1]$), scaling by $2\sqrt{p}$, and
rounding to integer. We then normalise to $\tilde{a}_p = a_p / (2\sqrt{p})$ to
match the ECQ data format, apply the same StandardScaler, and feed the result
into the CNN trained on XECQ$[0, 10000]$.

Grouping by predicted rank and plotting the average unnormalised $a_p$ reveals a
clear separation between classes, confirming the network has learned meaningful
$a_p$ bias patterns.

In [ ]:
# Generate synthetic a_p and classify with XECQ[0, 10000] model
fake_ap_raw = generate_fake_ap(500000, PRIME_COLUMNS, seed=42)
print(f'Generated {fake_ap_raw.shape[0]} synthetic sequences')

model = results['[0, 10000]']['model']
scaler = results['[0, 10000]']['scaler']
predictions = classify_fake_ap(model, scaler, fake_ap_raw, PRIME_COLUMNS, device)
print(f'Predicted distribution: {np.bincount(predictions, minlength=model.fc3.out_features)}')

In [ ]:
# Figure 10: Murmuration of synthetic data by predicted rank
# Plotting style adapted from classify_and_plot_fake_ranks.py
primes = np.array([float(p) for p in PRIME_COLUMNS])

rank_groups = {}
for rank in sorted(set(predictions)):
    rank_groups[rank] = fake_ap_raw[predictions == rank]

plt.figure(figsize=(12, 6))
cmap = plt.get_cmap('tab10')

for rank in sorted(rank_groups.keys()):
    data_arr = rank_groups[rank]
    if len(data_arr) < 100:
        continue
    avg_ap = np.mean(data_arr, axis=0)
    plt.plot(primes, avg_ap, '.', markersize=2, alpha=0.5, color=cmap(rank),
             label=f'Pred Rank {rank} (N={len(data_arr)})')

plt.axhline(y=0, color='r', linestyle='--', alpha=0.5)
plt.xlabel('Prime p')
plt.ylabel('Average $a_p$')
plt.title('Murmuration of Synthetic Elliptic Curves (by Predicted Rank)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()